In [ ]:
!pip install groq -q
print("Librarhy installed ")

Librarhy installed 


In [39]:
import os
from google.colab import userdata

# Set your Groq API key as an environment variable by fetching it from Colab's secrets manager
# Ensure you have added your GROQ_API_KEY to the secrets manager with the name 'GROQ_API_KEY'
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
print("GROQ_API_KEY has been set.")

SecretNotFoundError: Secret GROQ_API_KEY does not exist.

In [ ]:
import os
from groq import Groq

# Create the Groq client
client = Groq(api_key=os.environ["GROQ_API_KEY"])
# Groq(): Creates an instance of the Groq client class
# api_key: The parameter that authenticates your requests
# os.environ["GROQ_API_KEY"]: Reads the key we set above
# We store this in 'client' - we will use client.chat.completions.create() to call the AI

# Model we are using - same as Day 6 for continuity
MODEL = "llama-3.1-8b-instant"
# This is the specific LLM model we are using from Groq
# llama-3.1-8b-instant: A fast, efficient model by Meta, hosted on Groq
# 8b means 8 billion parameters - a medium-sized model
# 'instant' means it is optimized for speed

print("Groq client initialized successfully")
print(f"Using model: {MODEL}")

Groq client initialized successfully
Using model: llama-3.1-8b-instant


In [ ]:
import pandas as pd
import io

csv_data = """student_id,name,age,gender,subject,marks,attendance,grade
1,Aarav Sharma,20,Male,Mathematics,83,92,A
2,Priya Patel,21,Female,Science,76,85,D
4,Divya Krishnan,20,Female,Science,83,83,A
7,Karan Singh,22,Male,Mathematics,74,81,B
8,Ananya Gupta,21,Female,Programming,89,96,A
9,Vikram Reddy,20,Male,Science,20,77,B
10,Pooja Sharma,22,Female,Mathematics,85,72,D
11,Aditya Kumar,21,Male,Programming,97,99,A
21,Sunita Gupta,22,Female,Mathematics,87,90,A
29,Ravi Krishnan,21,Male,Programming,78,83,D
30,Sneha Roy,19,Female,Physics,88,95,A
31,Rahul Verma,20,Male,Chemistry,79,88,B
32,Deepika Singh,22,Female,Biology,91,97,A
33,Alok Kumar,21,Male,History,65,70,C
34,Neha Sharma,20,Female,Geography,72,80,B
35,Vikas Yadav,23,Male,Economics,80,89,A
36,Shweta Mishra,21,Female,Sociology,77,84,B
37,Pankaj Gupta,20,Male,Computer Science,94,98,A
38,Meena Devi,22,Female,Literature,68,75,C
39,Rajesh Patel,19,Male,Physical Education,85,91,A
40,Kavita Singh,20,Female,Art,70,78,C
41,Gaurav Sharma,21,Male,Music,82,90,A
42,Anjali Gupta,22,Female,Dance,75,82,B
43,Sanjay Kumar,19,Male,Engineering,90,96,A
44,Preeti Devi,20,Female,Medical Science,86,93,A
45,Amit Singh,21,Male,Law,73,81,B
46,Ritu Sharma,22,Female,Management,81,87,A
47,Vivek Kumar,19,Male,Statistics,78,85,B
48,Priyanka Singh,20,Female,Psychology,84,90,A
49,Manish Gupta,21,Male,Philosophy,60,65,D
50,Divya Sharma,22,Female,Environmental Science,89,94,A
51,Akash Yadav,19,Male,Political Science,67,72,C
52,Kiran Devi,20,Female,Journalism,71,79,B
53,Arjun Singh,21,Male,Architecture,92,97,A
54,Suman Gupta,22,Female,Fashion Design,74,80,B
55,Rohit Kumar,19,Male,Business Administration,88,95,A
56,Shilpa Sharma,20,Female,Human Resources,79,86,B
57,Aditi Singh,21,Female,Marketing,85,91,A
58,Harsh Patel,22,Male,Finance,87,93,A
59,Nisha Gupta,19,Female,Digital Marketing,80,88,A
60,Kartik Kumar,20,Male,Data Science,93,98,A
"""

df = pd.read_csv(io.StringIO(csv_data))

print(f"Dataset loaded: {len(df)} rows, {len(df.columns)} columns")
print("First 5 rows:")
print(df.head())

Dataset loaded: 41 rows, 8 columns
First 5 rows:
   student_id            name  age  gender      subject  marks  attendance  \
0           1    Aarav Sharma   20    Male  Mathematics     83          92   
1           2     Priya Patel   21  Female      Science     76          85   
2           4  Divya Krishnan   20  Female      Science     83          83   
3           7     Karan Singh   22    Male  Mathematics     74          81   
4           8    Ananya Gupta   21  Female  Programming     89          96   

  grade  
0     A  
1     D  
2     A  
3     B  
4     A  


In [ ]:
import sqlite3
import pandas as pd

# Step 2: Create a SQLite database and load the data

# Create a connection to a SQLite database file
# sqlite3.connect(): Creates a connection to a database.
# "college.db": The name of the database file to create (or open if it exists).
# If the file does not exist, SQLite creates it automatically.
# conn: The connection object - our link to the database.
conn = sqlite3.connect("college.db")

# Load the DataFrame into the database as a table called 'students'
# df.to_sql(): Converts a pandas DataFrame into a SQL table.
# "students": The name we give to the table in the database.
# conn: The database connection to write to.
# if_exists="replace": If a table named "students" already exists, delete it and recreate it.
#                      Other options: "fail" (raise error), "append" (add rows to existing table).
# index=False: Do not write the pandas row index (0,1,2...) as a column in the database.
df.to_sql("students", conn, if_exists="replace", index=False)

print("DataFrame loaded into 'students' table in college.db")

# Close the connection
conn.close()
print("Database connection closed.")

DataFrame loaded into 'students' table in college.db
Database connection closed.


In [ ]:
import sqlite3

# Function to get the database schema (table structure)
def get_schema(conn, table_name="students"):
    """
    This function reads the structure of a database table.
    It returns information about each column name and data type.

    Parameters:
    conn: The SQLite connection object (our link to the database)
    table_name: The name of the table to inspect (default: 'students')

    Returns:
    A formatted string describing the table structure
    """
    cursor = conn.cursor()
    cursor.execute(f"PRAGMA table_info({table_name})")
    columns_info = cursor.fetchall()

    if not columns_info:
        return f"Table '{table_name}' not found or has no columns."

    schema_string = f"Schema for table '{table_name}':\n"
    schema_string += "---------------------------------\n"
    for col in columns_info:
        cid, name, ctype, notnull, dflt_value, pk = col
        schema_string += f"Column Name: {name}, Data Type: {ctype}"
        if pk:
            schema_string += " (Primary Key)"
        if notnull:
            schema_string += " (NOT NULL)"
        schema_string += "\n"
    schema_string += "---------------------------------\n"
    return schema_string

In [ ]:
# Test the get_schema function
import sqlite3

# Establish a connection to the database
conn = sqlite3.connect("college.db")

# Get and print the schema for the 'students' table
students_schema = get_schema(conn, "students")
print(students_schema)

# Close the connection
conn.close()

Schema for table 'students':
---------------------------------
Column Name: student_id, Data Type: INTEGER
Column Name: name, Data Type: TEXT
Column Name: age, Data Type: INTEGER
Column Name: gender, Data Type: TEXT
Column Name: subject, Data Type: TEXT
Column Name: marks, Data Type: INTEGER
Column Name: attendance, Data Type: INTEGER
Column Name: grade, Data Type: TEXT
---------------------------------



In [ ]:
# Define the system prompt - this is the instruction we give to the LLM
system_prompt = f"""You are an expert SQL assistant.
You are connected to a SQLite database with the following structure:

{students_schema}

Rules you must follow:
1. Generate ONLY a valid SQLite SQL query.
2. Do not include any explanation or text - only the SQL query.
3. Do not use markdown code blocks. Return the raw SQL only.
"""

In [ ]:
# Define the user's question
user_question = "What are the names of students who have a grade of 'A' and scored more than 90 marks?"

# Call the new generate_sql_query function
# This will call the Groq API internally.
# Make sure your GROQ_API_KEY is valid to avoid AuthenticationError.
sql_query = generate_sql_query(user_question, system_prompt, client, MODEL)

if sql_query:
    print(f"User Question: {user_question}")
    print(f"Generated SQL Query:\n{sql_query}")
else:
    print("SQL query generation failed. Please check your API key and input.")


[STEP 2] Generating SQL query with Groq LLM for: What are the names of students who have a grade of 'A' and scored more than 90 marks? ...
Error generating SQL query: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}
SQL query generation failed. Please check your API key and input.


In [ ]:
import pandas as pd

def execute_sql(sql_query, conn):
    """
    Cleans the AI-generated SQL and executes it on the SQLite database.
    Returns the results as a pandas DataFrame.

    Parameters:
    sql_query (str): The SQL query string generated by the AI.
    conn (sqlite3.Connection): The SQLite connection object.

    Returns:
    pd.DataFrame: The results of the SQL query.
    """
    # Remove potential markdown or other unwanted characters from the start/end of the query
    clean_sql_query = sql_query.strip().strip('`').strip()

    try:
        # Execute the SQL query and fetch results into a pandas DataFrame
        results_df = pd.read_sql_query(clean_sql_query, conn)
        return results_df
    except Exception as e:
        print(f"Error executing SQL query: {e}")
        return pd.DataFrame()

In [ ]:
# S114627106333020260401161228

In [ ]:
import sqlite3

print("[Step 1] Executing SQL on the database...")

# Establish a connection to the database
conn = sqlite3.connect("college.db")

# Call our execute_sql() function
# It executes the generated SQL on the actual SQLite database
# It returns a result DataFrame or an empty DataFrame if there's an error

# Ensure sql_query is defined from the previous cell
if 'sql_query' in locals() and sql_query:
    results_df = execute_sql(sql_query, conn)
else:
    print("No SQL query available to execute.")
    results_df = pd.DataFrame()

# Close the connection
conn.close()

if not results_df.empty:
    print("\nSQL Execution Successful! Results:")
    print(results_df)
else:
    print("\nSQL Execution failed or returned no results.")
    print("Please ensure 'sql_query' is correctly populated and the SQL query is valid.")

# Note: The 'sql_query' variable is populated from the Groq API call.
# If you encounter an 'Invalid API Key' error, please ensure your GROQ_API_KEY is valid to get a proper 'sql_query'.

[Step 1] Executing SQL on the database...
No SQL query available to execute.

SQL Execution failed or returned no results.
Please ensure 'sql_query' is correctly populated and the SQL query is valid.


In [ ]:
import pandas as pd
from groq import Groq # Ensure Groq is imported if not globally available, though it should be from earlier cells.

def generate_sql_query(user_question: str, system_prompt: str, client: Groq, model: str) -> str:
    """
    Generates a SQLite SQL query based on a natural language question and database schema
    using the Groq LLM.

    Args:
        user_question (str): The natural language question to convert to SQL.
        system_prompt (str): The system prompt including the database schema.
        client (Groq): The initialized Groq client object.
        model (str): The name of the LLM model to use (e.g., "llama-3.1-8b-instant").

    Returns:
        str: The generated SQL query string.
    """
    print(f"\n[STEP 2] Generating SQL query with Groq LLM for: {user_question} ...")
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_question}
            ],
            temperature=0.0
        )
        sql_query = response.choices[0].message.content.strip()
        print("Generated SQL Query:")
        print(sql_query)
        return sql_query
    except Exception as e:
        print(f"Error generating SQL query: {e}")
        # Return an empty string if SQL generation fails, so subsequent steps can handle it.
        return ""